# 🏥 MyHealth Decoder — Patient Advocate

**Med-Gemma Impact Challenge Submission**

| Item | Details |
|------|--------|
| **Use Case** | Patient Advocate — Decode medical reports into plain English |
| **Model** | MedGemma Instruct 4B (HAI-DEF) |
| **Team** | Kiza |
| **Framework** | Transformers + Gradio |
| **Target Awards** | Main Track + Edge AI Prize |

---

## 1. Setup & Dependencies

In [ ]:
# Install dependencies
!pip install -q transformers accelerate gradio

import torch
import os

# Environment check
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected. Model will run on CPU (slow).")

## 2. Load MedGemma

We use `google/medgemma-4b-it` — the instruction-tuned 4B parameter multimodal variant,
optimized for medical text understanding.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/medgemma-4b-it"

print(f"⏳ Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else "cpu",
)
print(f"✅ Model loaded on {model.device}")

## 3. Define Prompting Strategy

The system prompt defines MyHealth Decoder's persona: empathetic, thorough, and clear.
It structures output into 4 sections: Summary, Key Findings, Implications, and Doctor Questions.

In [ ]:
SYSTEM_PROMPT = """You are MyHealth Decoder, an empathetic and thorough medical advocate AI.
Your role is to help patients understand their medical reports.

Rules:
1. Explain ALL medical terms in simple, 5th-grade English.
2. Never diagnose or override the doctor's assessment.
3. Always recommend discussing findings with the treating physician.
4. Highlight what is NORMAL vs what needs ATTENTION.
5. Generate 3-5 specific questions the patient should ask their doctor.
6. Use a warm, reassuring but honest tone.
7. Structure your response with: Simple Summary, Key Findings, What This Means, Questions to Ask.
"""

def decode_report(report_text: str) -> str:
    """Decode a medical report into patient-friendly language using MedGemma."""
    prompt = f"""<start_of_turn>user
{SYSTEM_PROMPT}

Please analyze this medical report and explain it to me as a patient:

---
{report_text}
---

Provide:
1. **Simple Summary** — What does this report say in plain English?
2. **Key Findings** — Break down each finding (what's normal, what needs attention).
3. **What This Means For You** — Practical implications.
4. **Questions to Ask Your Doctor** — 3-5 specific, relevant questions.
<end_of_turn>
<start_of_turn>model
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
        )
    
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return response

print("✅ decode_report() ready")

## 4. Test with Sample Reports

We test MyHealth Decoder on 4 diverse report types:
- Radiology (Chest X-Ray)
- Lab Work (CBC + Metabolic Panel)
- Pathology (Breast Biopsy)
- CT Scan (Pulmonary Follow-up)

In [ ]:
# Test 1: Chest X-Ray
xray_report = """
CHEST X-RAY, PA AND LATERAL
CLINICAL INDICATION: Cough, shortness of breath.
FINDINGS:
The lungs are clear bilaterally. No focal consolidation, pleural effusion, or pneumothorax.
The cardiac silhouette is mildly enlarged with a cardiothoracic ratio of 0.55 (normal < 0.5).
Mild degenerative change of the thoracic spine.
IMPRESSION:
1. Mild cardiomegaly. Recommend echocardiogram.
2. No acute pulmonary disease.
"""

print("🩺 REPORT: Chest X-Ray")
print("=" * 60)
explanation = decode_report(xray_report)
print(explanation)
print("\n" + "=" * 60)

In [ ]:
# Test 2: Blood Work
blood_report = """
COMPLETE BLOOD COUNT WITH DIFFERENTIAL
WBC: 11.2 x10^3/uL (H) [Reference: 4.5-11.0]
Hemoglobin: 14.2 g/dL [Reference: 13.5-17.5]
Platelets: 245 x10^3/uL [Reference: 150-400]

COMPREHENSIVE METABOLIC PANEL
Glucose: 126 mg/dL (H) [Reference: 70-100]
HbA1c: 6.8% (H) [Reference: <5.7]
ALT: 45 U/L (H) [Reference: 7-35]
AST: 38 U/L (H) [Reference: 10-34]
"""

print("🩺 REPORT: Blood Work")
print("=" * 60)
explanation = decode_report(blood_report)
print(explanation)

In [ ]:
# Test 3: Pathology Report
path_report = """
SURGICAL PATHOLOGY REPORT
SPECIMEN: Excisional biopsy, left breast, 2 o'clock position.
Invasive ductal carcinoma, grade 2 (of 3). Tumor size: 1.4 cm.
Margins: Negative, closest margin 0.3 cm.
Lymphovascular invasion: Not identified.
ER: Positive (95%, strong). PR: Positive (80%, moderate). HER2: Negative (1+). Ki-67: 15%.
IMPRESSION: Invasive ductal carcinoma, pT1c. ER+/PR+/HER2-. Margins negative.
"""

print("🩺 REPORT: Pathology")
print("=" * 60)
explanation = decode_report(path_report)
print(explanation)

## 5. Interactive Demo with Gradio

Launch an interactive interface where users can paste any medical report and get a plain-English explanation.

In [ ]:
import gradio as gr

SAMPLE_REPORTS = {
    "Chest X-Ray": xray_report.strip(),
    "Blood Work": blood_report.strip(),
    "Pathology": path_report.strip(),
}

def load_sample(name):
    return SAMPLE_REPORTS.get(name, "")

def analyze(report_text, _):
    if not report_text or not report_text.strip():
        return "⚠️ Please paste a medical report."
    return decode_report(report_text)

with gr.Blocks(title="MyHealth Decoder") as demo:
    gr.HTML("<h1>🏥 MyHealth Decoder</h1><p>Powered by MedGemma (HAI-DEF)</p>")
    
    with gr.Row():
        with gr.Column():
            dropdown = gr.Dropdown(choices=list(SAMPLE_REPORTS.keys()), label="Sample Reports")
            report = gr.Textbox(label="Medical Report", lines=12, placeholder="Paste report here...")
            btn = gr.Button("🔍 Decode My Report", variant="primary")
        with gr.Column():
            output = gr.Markdown(label="Explanation")
    
    dropdown.change(load_sample, [dropdown], [report])
    btn.click(analyze, [report, dropdown], [output])

demo.launch(share=True)

## 6. Technical Notes

### Model Choice
- **MedGemma 4B Instruct**: Chosen for the instruction-following capability and medical domain fine-tuning.
- 4B parameter size enables deployment on consumer GPUs and edge devices.

### Privacy-First Design
- All inference runs **locally** — no patient data leaves the device.
- No external API calls during inference.
- Suitable for HIPAA-adjacent workflows.

### Edge AI Potential
- 4B model fits in ~8GB VRAM (bfloat16)
- Quantization (4-bit GPTQ/GGUF) reduces to ~3GB → runs on smartphones/tablets
- Offline-capable for rural/underserved healthcare settings

### Limitations
- Does NOT provide medical diagnoses or treatment recommendations
- Best used as a supplement to, not replacement for, physician consultation
- Performance varies by report complexity and medical domain